# AAR Legibility PVG: experiments (v4, Brandon's fixes)

Same as v3, plus `apply_fixes.py` from Brandon's main (6 Oct). It pins the code to the fork commit 51940d1, then adds five changes: failed helpful samples are dropped from verifier training and each round is class-balanced; Qwen-written SOUND rows with perturbed UNSOUND twins; 5 more real records; a supervised warm start of both provers before round 1; a +0.1 role bonus on the prover reward. Each one has a config switch, printed at the end of the fixes cell.

Runtime: A100. About 15 minutes of setup (model-written rows), a few minutes of warm start per seed, then 6 to 9 minutes per round.

## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/gunasti2002/Automated_R_leg_NoAPI.git"
BRANCH    = "main"
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
!git -C {CLONE_DIR} log --oneline -3


In [ ]:
# Colab ships torchao 0.10, which peft refuses (needs >= 0.16). Nothing here uses it: remove it.
!pip uninstall -y -q torchao
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate
import importlib; importlib.invalidate_caches()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


### Brandon's fixes

Pins the project code to 51940d1, patches it, rebuilds the dataset with the 10 real records and writes the Qwen-written rows (GPU, about 15 minutes). In the output, check that every patch line starts with `+` and that both model-written files have rows. The data cell below rebuilds the same dataset, which is harmless.

In [ ]:
!python apply_fixes.py
!git -C {CLONE_DIR} status --short | head -20

In [ ]:
# Brandon's change 3: a stronger learning signal for the prover. Subprocesses inherit this.
import os, json
OVERRIDES = {
    "prover_lr": 1e-4,                 # was 2e-5
    "prover_num_generations": 8,       # was 4 samples per record
    "findings_per_round": 16,          # was 8 records per round
    "prover_kl_penalty": 0.05,         # was 0.02
    # Both roles gated by the rule check: helpful earns P(sound) only if it passes,
    # sneaky only if it fails. Matches the verifier labels (sneaky write-ups that pass
    # the rule check are dropped from verifier training, so they should earn nothing).
    "prover_reward_mode": "correctness_gated",
}
os.environ["PVG_CONFIG_OVERRIDES"] = json.dumps(OVERRIDES)

import sys; sys.path.insert(0, ".")
from config import PVGConfig
cfg = PVGConfig()
print({k: getattr(cfg, k) for k in OVERRIDES})
print("samples per round:", 2 * cfg.findings_per_round * cfg.prover_num_generations)


## 2. Data

120 honest/sneaky pairs, 20% held out by record, seed 42 (unchanged).

In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json | tail -4


## 3. Does the untrained prover write findings that pass the rule check?

Two minutes, no training. This is the check for change 1. If helpful fidelity is below 0.5
here, the helpful prover will earn almost no reward and the rest of the run repeats the
26 Sep failure, so fix the prompt before going further. The script prints why the failing
write-ups fail and shows two of them.

This is base Qwen, before the warm start. The pilot log prints the warm-started number as `[sft] after warm start: helpful fidelity`, and that one should be at least 0.5.

In [ ]:
!python eval/prover_fidelity.py --records 8 --generations 4 --min-helpful 0.5


In [ ]:
# Verifier spot check (untrained) and the two baselines on the held-out split.
from training import pvg_loop
spot_rows = pvg_loop.load_dataset(cfg.spot_check_set_path)
report = pvg_loop.run_spot_check(cfg, spot_set_rows=spot_rows, enforce_set=False)
!python eval/baselines.py --out eval/baselines_heldout.json | grep -E "^\["


In [ ]:
# Drop the notebook's verifier so the training subprocesses get the full card.
import gc, sys, torch
if "training.pvg_loop" in sys.modules:
    sys.modules["training.pvg_loop"]._VERIFIER_CACHE.clear()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")
assert free / total > 0.9, "Something is still holding GPU memory — restart the runtime."


## 4. Pilot: seed 42, 3 rounds, then check the four targets

The pilot writes to the same checkpoint folder as the full run, so the full run in
section 5 continues seed 42 from round 4 instead of repeating these rounds.
Exit 2 = the spot-check gate refused to train. Exit 3 = the seed aborted itself.

In [ ]:
SEEDS          = [42, 43, 44]
TOTAL_ROUNDS   = 10
PILOT_ROUNDS   = 3
CHECKPOINT_DIR = "training/checkpoint"


In [ ]:
!python run_experiment.py --seeds 42 --rounds {PILOT_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR} --no-plot


In [ ]:
# Brandon's four targets, checked per seed from round_history.jsonl.
import json
from pathlib import Path

def load_history(ckpt):
    out = {}
    for f in sorted(Path(ckpt).glob("seed_*/round_history.jsonl")):
        rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
        if rows: out[rows[0].get("seed", f.parent.name)] = rows
    return out

def check_targets(ckpt):
    hist = load_history(ckpt)
    if not hist:
        print("No completed rounds yet."); return
    for seed, rows in hist.items():
        r1, last = rows[0], rows[-1]
        fid = [r["role_fidelity_helpful"] for r in rows]
        gap = [r["reward_gap"] for r in rows]
        bias = [r.get("verifier_logit_bias") or 0.0 for r in rows]
        acc = [r["helpful_prover_accuracy"] for r in rows]
        share = [r.get("verifier_train_sound_share") for r in rows]
        checks = [
            ("helpful fidelity >= 0.5 in round 1 and not falling", fid[0] >= 0.5 and fid[-1] >= fid[0] - 0.1,
             "  ".join(f"{x:.2f}" for x in fid)),
            ("reward gap stays positive", min(gap) > 0, "  ".join(f"{x:+.2f}" for x in gap)),
            ("logit bias within +/-3", max(abs(b) for b in bias) <= 3.0, "  ".join(f"{x:+.1f}" for x in bias)),
            ("helpful accuracy does not crash after round 1", min(acc[1:] or acc) >= 0.5,
             "  ".join(f"{x:.2f}" for x in acc)),
        ]
        print(f"seed {seed}: {len(rows)} round(s)" + ("  ABORTED: " + last.get("notes", "") if last.get("aborted") else ""))
        for name, ok, vals in checks:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name:52s} {vals}")
        print("         SOUND share of verifier training pool      ", "  ".join("n/a" if s is None else f"{s:.2f}" for s in share))
        print("         held-out accuracy / AUROC                  ",
              "  ".join(f"{r['heldout_accuracy']:.2f}/{r['heldout_auroc']:.2f}" for r in rows))

check_targets(CHECKPOINT_DIR)


In [ ]:
# What the helpful prover actually wrote in the latest round: failures with the rule check's reasons.
import json
from pathlib import Path
files = sorted(Path(CHECKPOINT_DIR).glob("seed_42/samples/round_*.jsonl"))
if files:
    rows = [json.loads(l) for l in files[-1].read_text().splitlines() if l.strip()]
    h = [r for r in rows if r["label"] == "helpful"]
    fails = [r for r in h if not r["rule_check_passes"]]
    print(f"{files[-1].name}: {len(h)} helpful samples, {len(fails)} fail the rule check\n")
    for r in fails[:4]:
        print(json.dumps(r["experiment"]["metrics"]))
        print("  issues:", r["consistency_issues"])
        print("  " + r["summary"][:600].replace("\n", " ") + "\n")
else:
    print("No sample files yet.")


## 5. Full run and results

Continues seed 42 from where the pilot stopped, then runs seeds 43 and 44.
Safe to interrupt; re-run the cell to resume.

In [ ]:
seeds = " ".join(str(s) for s in SEEDS)
!python run_experiment.py --seeds {seeds} --rounds {TOTAL_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR}


In [ ]:
import subprocess, sys
from pathlib import Path
png = Path(CHECKPOINT_DIR) / "pvg_curves.png"
subprocess.run([sys.executable, "eval/plot_rounds.py", "--checkpoint-dir", CHECKPOINT_DIR, "--out", str(png)])
if png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(png)))
check_targets(CHECKPOINT_DIR)


In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for f in sorted(Path(CHECKPOINT_DIR).glob("seed_*/round_history.jsonl")):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
if df.empty:
    print("No completed rounds yet.")
cols = ["seed", "round_number", "helpful_prover_accuracy", "verifier_robustness", "role_fidelity_helpful",
        "role_fidelity_sneaky", "reward_gap", "helpful_mean_reward", "sneaky_mean_reward",
        "heldout_accuracy", "heldout_auroc", "heldout_accept_rate", "verifier_logit_bias",
        "verifier_train_sound_share", "verifier_relabeled_helpful", "verifier_dropped_sneaky", "aborted", "elapsed_s"]
df[[c for c in cols if c in df.columns]]


### Save the results before the runtime ends

Colab's disk is wiped when the runtime is deleted, and the per-round sample files are the
only record of what the provers wrote. This zips the histories, samples, plot and baselines
(not the model adapters) and downloads the archive.

In [ ]:
import shutil, os
from pathlib import Path
out = Path("/content/pvg_results"); shutil.rmtree(out, ignore_errors=True); out.mkdir()
for f in Path(CHECKPOINT_DIR).rglob("*"):
    if f.is_file() and (f.suffix in (".jsonl", ".json", ".png")) and "verifier" not in f.parts \
            and "helpful_prover" not in f.parts and "sneaky_prover" not in f.parts and f.name != "replay_buffer.json":
        dest = out / f.relative_to(CHECKPOINT_DIR); dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dest)
for f in ("eval/baselines_heldout.json", "eval/prover_fidelity.json", "data/dataset_manifest.json"):
    if Path(f).exists(): shutil.copy(f, out / Path(f).name)
shutil.make_archive("/content/pvg_results", "zip", out)
print("files:", sum(1 for _ in out.rglob("*") if _.is_file()), "| archive MB:", round(os.path.getsize("/content/pvg_results.zip") / 1e6, 2))
from google.colab import files
files.download("/content/pvg_results.zip")
